In [36]:
# Cell 1 — Imports, connection, schema introspection
import psycopg2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.4f}'.format)

# --- Connection
conn = psycopg2.connect(
    host='127.0.0.1', port=5455, dbname='postgres',
    user='postgres', password='postgres'
)
cur = conn.cursor()
print("Connected.")

# --- Schema introspection: int_game_team_features
cur.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'int'
      AND table_name = 'int_game_team_features'
    ORDER BY ordinal_position
""")
rows = cur.fetchall()
cols = [d[0] for d in cur.description]
schema_gtf = pd.DataFrame(rows, columns=cols)
print(f"\nint_game_team_features — {len(schema_gtf)} columns")
print(schema_gtf.to_string(index=False))

Connected.

int_game_team_features — 28 columns
                column_name data_type
                    game_id    bigint
                     season   integer
                       week   integer
                  game_date      date
                  team_name      text
                   opponent      text
              points_scored   integer
             points_allowed   integer
                        win   integer
           off_epa_per_play   numeric
   def_epa_per_play_allowed   numeric
    close_game_epa_per_play   numeric
      close_game_play_count    bigint
close_game_def_epa_per_play   numeric
  close_game_def_play_count    bigint
                game_script      text
     game_script_avg_margin   numeric
          last3_off_epa_avg   numeric
              last3_win_pct   numeric
    last3_points_scored_avg   numeric
          last3_def_epa_avg   numeric
   last3_points_allowed_avg   numeric
       days_since_last_game   integer
 opp_sp_rating_at_game_time   numeric
  

In [37]:
# Cell 2 — Load FBS conference game population + conference distribution + game_script value inspection
query = """
    SELECT
        f.game_id,
        f.season,
        f.week,
        f.team_name,
        f.opponent,
        f.points_scored,
        f.points_allowed,
        f.game_script,
        f.game_script_avg_margin,
        f.close_game_play_count,
        f.close_game_def_play_count,
        f.close_game_epa_per_play,
        f.close_game_def_epa_per_play,
        f.off_epa_per_play,
        f.def_epa_per_play_allowed,
        f.opp_sp_rating_at_game_time,
        f.pregame_elo,
        f.opponent_pregame_elo,
        f.excitement_index,
        s.conference,
        s.sp_rating
    FROM int.int_game_team_features f
    INNER JOIN int.int_team_season_features s
        ON f.team_name = s.team_name
        AND f.season = s.season
    WHERE f.season IN (2022, 2023, 2024)
      AND s.conference != 'FBS Independents'
"""

cur.execute(query)
rows = cur.fetchall()
cols = [d[0] for d in cur.description]
df_raw = pd.DataFrame(rows, columns=cols)

# --- Cast numerics
numeric_cols = [
    'points_scored', 'points_allowed', 'game_script_avg_margin',
    'close_game_play_count', 'close_game_def_play_count',
    'close_game_epa_per_play', 'close_game_def_epa_per_play',
    'off_epa_per_play', 'def_epa_per_play_allowed',
    'opp_sp_rating_at_game_time', 'pregame_elo', 'opponent_pregame_elo',
    'excitement_index', 'sp_rating'
]
df_raw[numeric_cols] = df_raw[numeric_cols].astype(float)

# --- Derive targets
df_raw['point_differential'] = df_raw['points_scored'] - df_raw['points_allowed']
df_raw['total_points']       = df_raw['points_scored'] + df_raw['points_allowed']

print(f"Total team-game rows: {len(df_raw):,}")
print(f"Unique games:         {df_raw['game_id'].nunique():,}")
print(f"Seasons:              {sorted(df_raw['season'].unique())}")

# --- FBS integrity check
print("\n--- Conference distribution (must not contain FBS Independents) ---")
conf_dist = df_raw['conference'].value_counts()
print(conf_dist.to_string())

assert 'FBS Independents' not in conf_dist.index, "STOP: FBS Independents leaked through"
print("\n✓ FBS Independents not present — integrity check passed")

# --- game_script distinct values
print("\n--- game_script distinct values and counts ---")
print(df_raw['game_script'].value_counts(dropna=False).to_string())

# --- Null check on controls
controls = ['close_game_epa_per_play', 'close_game_def_epa_per_play', 'sp_rating']
null_counts = df_raw[controls].isnull().sum()
print("\n--- Null counts on controls ---")
print(null_counts.to_string())

Total team-game rows: 4,663
Unique games:         2,567
Seasons:              [np.int64(2022), np.int64(2023), np.int64(2024)]

--- Conference distribution (must not contain FBS Independents) ---
conference
Big Ten              558
ACC                  543
SEC                  534
Sun Belt             507
Big 12               486
American Athletic    474
Mountain West        440
Mid-American         438
Conference USA       367
Pac-12               316

✓ FBS Independents not present — integrity check passed

--- game_script distinct values and counts ---
game_script
competitive      2646
deficit           815
comfortable       728
dominant          259
large_deficit     215

--- Null counts on controls ---
close_game_epa_per_play        6
close_game_def_epa_per_play    6
sp_rating                      0


In [38]:
# Cell 3 — Rebuild load: conference games only, both teams FBS non-independent
query = """
    SELECT
        f.game_id,
        f.season,
        f.week,
        f.team_name,
        f.opponent,
        f.points_scored,
        f.points_allowed,
        f.game_script,
        f.game_script_avg_margin,
        f.close_game_play_count,
        f.close_game_def_play_count,
        f.close_game_epa_per_play,
        f.close_game_def_epa_per_play,
        f.off_epa_per_play,
        f.def_epa_per_play_allowed,
        f.opp_sp_rating_at_game_time,
        f.pregame_elo,
        f.opponent_pregame_elo,
        f.excitement_index,
        s.conference,
        s.sp_rating
    FROM int.int_game_team_features f
    INNER JOIN int.int_team_season_features s
        ON f.team_name = s.team_name
        AND f.season = s.season
    -- opponent must also be FBS non-independent
    INNER JOIN int.int_team_season_features s_opp
        ON f.opponent = s_opp.team_name
        AND f.season = s_opp.season
        AND s_opp.conference != 'FBS Independents'
    -- conference game filter
    INNER JOIN raw.games g
        ON f.game_id = g.id
        AND g.conference_game = TRUE
    WHERE f.season IN (2022, 2023, 2024)
      AND s.conference != 'FBS Independents'
"""

cur.execute(query)
rows = cur.fetchall()
cols = [d[0] for d in cur.description]
df_raw = pd.DataFrame(rows, columns=cols)

# --- Cast numerics
numeric_cols = [
    'points_scored', 'points_allowed', 'game_script_avg_margin',
    'close_game_play_count', 'close_game_def_play_count',
    'close_game_epa_per_play', 'close_game_def_epa_per_play',
    'off_epa_per_play', 'def_epa_per_play_allowed',
    'opp_sp_rating_at_game_time', 'pregame_elo', 'opponent_pregame_elo',
    'excitement_index', 'sp_rating'
]
df_raw[numeric_cols] = df_raw[numeric_cols].astype(float)

# --- Derive targets
df_raw['point_differential'] = df_raw['points_scored'] - df_raw['points_allowed']
df_raw['total_points']       = df_raw['points_scored'] + df_raw['points_allowed']

print(f"Total team-game rows: {len(df_raw):,}")
print(f"Unique games:         {df_raw['game_id'].nunique():,}")
print(f"Seasons:              {sorted(df_raw['season'].unique())}")

# --- FBS integrity check
print("\n--- Conference distribution (must not contain FBS Independents) ---")
conf_dist = df_raw['conference'].value_counts()
print(conf_dist.to_string())
assert 'FBS Independents' not in conf_dist.index, "STOP: FBS Independents leaked through"
print("\n✓ FBS Independents not present — integrity check passed")

# --- game_script value counts
print("\n--- game_script distinct values and counts ---")
print(df_raw['game_script'].value_counts(dropna=False).to_string())

# --- Null check on controls
controls = ['close_game_epa_per_play', 'close_game_def_epa_per_play', 'sp_rating']
null_counts = df_raw[controls].isnull().sum()
print("\n--- Null counts on controls ---")
print(null_counts.to_string())

Total team-game rows: 3,214
Unique games:         1,607
Seasons:              [np.int64(2022), np.int64(2023), np.int64(2024)]

--- Conference distribution (must not contain FBS Independents) ---
conference
Big Ten              420
Big 12               366
ACC                  364
SEC                  358
Sun Belt             342
American Athletic    320
Mid-American         294
Mountain West        282
Conference USA       246
Pac-12               222

✓ FBS Independents not present — integrity check passed

--- game_script distinct values and counts ---
game_script
competitive      1966
deficit           594
comfortable       438
large_deficit     134
dominant           82

--- Null counts on controls ---
close_game_epa_per_play        6
close_game_def_epa_per_play    6
sp_rating                      0


In [40]:
# Cell 4 — Build analysis dataframe: drop nulls, matchup frame, tiers, trajectory, ordinal encoding

# --- Drop rows with null controls
df = df_raw.dropna(subset=['close_game_epa_per_play', 'close_game_def_epa_per_play']).copy()
print(f"Rows after dropping null controls: {len(df):,}")
print(f"Unique games after drop:           {df['game_id'].nunique():,}")

# --- Canonical assign_tier
P4_CONFERENCES = {"ACC", "Big 12", "Big Ten", "SEC"}

def assign_tier(row):
    if row["team_name"] == "Notre Dame":
        return "P4"
    if row["team_name"] == "UConn":
        return "G5"
    if row["conference"] in P4_CONFERENCES:
        return "P4"
    return "G5"

df['tier'] = df.apply(assign_tier, axis=1)

# --- Ordinal encoding for game_script
# Scale: dominant (led big) → comfortable → competitive → deficit → large_deficit (trailed big)
GAME_SCRIPT_ORDINAL = {
    'dominant':     2,
    'comfortable':  1,
    'competitive':  0,
    'deficit':     -1,
    'large_deficit':-2,
}
df['game_script_ordinal'] = df['game_script'].map(GAME_SCRIPT_ORDINAL)
print("\ngame_script ordinal mapping:")
print(df.groupby('game_script')['game_script_ordinal'].first().sort_values(ascending=False).to_string())

# --- Conference game sequence number per team per season
# Use week as proxy for ordering within season
df = df.sort_values(['team_name', 'season', 'week'])
df['conf_game_seq'] = df.groupby(['team_name', 'season']).cumcount() + 1

print("\nConference game sequence distribution:")
print(df['conf_game_seq'].value_counts().sort_index().to_string())

# --- Trajectory bucket
def assign_trajectory(seq):
    if seq == 1:
        return 'game_1'
    elif seq <= 4:
        return 'games_2_4'
    elif seq <= 8:
        return 'games_5_8'
    else:
        return 'games_9_12'

df['trajectory'] = df['conf_game_seq'].apply(assign_trajectory)
print("\nTrajectory bucket distribution:")
print(df['trajectory'].value_counts().to_string())

# --- Build matchup frame: one row per game
# Home team = team appearing as home in raw.games
# We derive point_differential and total_points from team perspective already
# For game-level frame: use the home-team row as the anchor
# Identify home side: join back through raw.games would require another query.
# Instead: for each game_id take the row where point_differential is positive
# as a tiebreaker — actually we need explicit home/away.
# Safe approach: deduplicate on game_id keeping the first alphabetical team_name
# for stable dedup, then compute game-level deltas from paired rows.

# Pair rows by game_id
df_home = df.copy()
df_away = df.copy()

# For each game we need team A vs team B — merge on game_id where team != opponent
paired = df_home.merge(
    df_away[['game_id', 'team_name', 'conference', 'tier',
             'close_game_epa_per_play', 'close_game_def_epa_per_play',
             'sp_rating', 'game_script', 'game_script_ordinal',
             'game_script_avg_margin', 'close_game_play_count',
             'close_game_def_play_count', 'pregame_elo',
             'point_differential', 'total_points', 'trajectory', 'conf_game_seq']],
    left_on=['game_id', 'opponent'],
    right_on=['game_id', 'team_name'],
    suffixes=('_a', '_b')
)

# Keep one row per game: team_name_a < team_name_b (alphabetical dedup)
paired = paired[paired['team_name_a'] < paired['team_name_b']].copy()
print(f"\nMatchup frame rows (one per game): {len(paired):,}")

# --- Game-level deltas (team_a perspective)
paired['close_game_epa_delta']     = paired['close_game_epa_per_play_a']     - paired['close_game_epa_per_play_b']
paired['close_game_def_epa_delta'] = paired['close_game_def_epa_per_play_a'] - paired['close_game_def_epa_per_play_b']
paired['sp_rating_delta']          = paired['sp_rating_a']                   - paired['sp_rating_b']
paired['elo_delta']                = paired['pregame_elo_a']                 - paired['pregame_elo_b']
paired['game_script_ordinal_delta']= paired['game_script_ordinal_a']         - paired['game_script_ordinal_b']
paired['game_script_avg_margin_delta'] = paired['game_script_avg_margin_a']  - paired['game_script_avg_margin_b']
paired['close_game_play_count_delta']  = paired['close_game_play_count_a']   - paired['close_game_play_count_b']

# Targets from team_a perspective
paired['point_differential'] = paired['point_differential_a']
paired['total_points']       = paired['total_points_a']   # same for both sides
paired['abs_residual']       = paired['point_differential_a'].abs()  # placeholder for moneyline variance

# Trajectory — use team_a trajectory (both sides same game, seq may differ slightly)
paired['trajectory'] = paired['trajectory_a']
paired['conference'] = paired['conference_a']
paired['tier']       = paired['tier_a']

print("\nNull check on matchup-frame deltas:")
delta_cols = ['close_game_epa_delta', 'close_game_def_epa_delta', 'sp_rating_delta',
              'game_script_ordinal_delta', 'game_script_avg_margin_delta',
              'close_game_play_count_delta']
print(paired[delta_cols].isnull().sum().to_string())

print("\nTrajectory distribution in matchup frame:")
print(paired['trajectory'].value_counts().to_string())

print("\nTier distribution:")
print(paired['tier'].value_counts().to_string())

Rows after dropping null controls: 3,208
Unique games after drop:           1,604

game_script ordinal mapping:
game_script
dominant         2
comfortable      1
competitive      0
deficit         -1
large_deficit   -2

Conference game sequence distribution:
conf_game_seq
1     384
2     382
3     382
4     382
5     382
6     382
7     382
8     365
9     150
10     17

Trajectory bucket distribution:
trajectory
games_5_8     1511
games_2_4     1146
game_1         384
games_9_12     167

Matchup frame rows (one per game): 1,604

Null check on matchup-frame deltas:
close_game_epa_delta            0
close_game_def_epa_delta        0
sp_rating_delta                 0
game_script_ordinal_delta       0
game_script_avg_margin_delta    0
close_game_play_count_delta     0

Trajectory distribution in matchup frame:
trajectory
games_5_8     765
games_2_4     571
game_1        187
games_9_12     81

Tier distribution:
tier
G5    850
P4    754


In [41]:
# Cell 5 — Partial r helpers (used for all signal tests)

from sklearn.linear_model import LinearRegression
from scipy import stats

MIN_N_FULL   = 30   # minimum n for full-population verdict
MIN_N_STRAT  = 15   # minimum n for stratified verdict

def partial_r(df, feature, target, controls):
    """
    Compute partial correlation of feature with target after regressing out controls.
    Returns (partial_r, p_value, n). Returns (nan, nan, n) if insufficient data.
    """
    cols = [feature] + controls + [target]
    sub = df[cols].dropna()
    n = len(sub)
    if n < MIN_N_FULL:
        return np.nan, np.nan, n

    X_ctrl = sub[controls].values
    y      = sub[target].values
    f      = sub[feature].values

    # Residualize target on controls
    reg_y = LinearRegression().fit(X_ctrl, y)
    resid_y = y - reg_y.predict(X_ctrl)

    # Residualize feature on controls
    reg_f = LinearRegression().fit(X_ctrl, f)
    resid_f = f - reg_f.predict(X_ctrl)

    r, p = stats.pearsonr(resid_f, resid_y)
    return r, p, n


def partial_r_table(df, features, targets, controls, group_col=None, group_vals=None):
    """
    Run partial_r for each feature × target combination.
    If group_col and group_vals provided, run within each group.
    Returns a DataFrame of results.
    """
    records = []
    groups = [(None, df)] if group_col is None else [
        (gv, df[df[group_col] == gv]) for gv in group_vals
    ]
    for gval, gdf in groups:
        for feat in features:
            for tgt in targets:
                r, p, n = partial_r(gdf, feat, tgt, controls)
                records.append({
                    'group':   gval if gval is not None else 'full_population',
                    'feature': feat,
                    'target':  tgt,
                    'partial_r': round(r, 4) if not np.isnan(r) else np.nan,
                    'p_value':   round(p, 4) if not np.isnan(p) else np.nan,
                    'n':         n,
                    'signal':    (abs(r) >= 0.10 and p < 0.05) if not np.isnan(r) else False,
                })
    return pd.DataFrame(records)


def moneyline_variance_partial_r(df, feature, controls, group_col=None, group_vals=None):
    """
    Moneyline variance signal: partial r of feature with abs(residual of
    point_differential after controls). Higher abs residual = more outcome uncertainty.
    Returns DataFrame with group, feature, partial_r, p_value, n, signal.
    """
    # First residualize point_differential on controls to get base residuals
    records = []
    groups = [(None, df)] if group_col is None else [
        (gv, df[df[group_col] == gv]) for gv in group_vals
    ]
    for gval, gdf in groups:
        cols = [feature] + controls + ['point_differential']
        sub = gdf[cols].dropna()
        n = len(sub)
        if n < MIN_N_FULL:
            records.append({'group': gval or 'full_population', 'feature': feature,
                            'partial_r': np.nan, 'p_value': np.nan, 'n': n, 'signal': False})
            continue
        X_ctrl = sub[controls].values
        y      = sub['point_differential'].values
        f      = sub[feature].values

        reg_y  = LinearRegression().fit(X_ctrl, y)
        resid_y = np.abs(y - reg_y.predict(X_ctrl))

        reg_f  = LinearRegression().fit(X_ctrl, f)
        resid_f = f - reg_f.predict(X_ctrl)

        r, p = stats.pearsonr(resid_f, resid_y)
        records.append({
            'group':     gval or 'full_population',
            'feature':   feature,
            'partial_r': round(r, 4),
            'p_value':   round(p, 4),
            'n':         n,
            'signal':    abs(r) >= 0.10 and p < 0.05,
        })
    return pd.DataFrame(records)


# --- Define feature sets and controls
SCRIPT_FEATURES = [
    'game_script_ordinal_delta',
    'game_script_avg_margin_delta',
    'close_game_play_count_delta',
]

TARGETS_SPREAD  = ['point_differential']
TARGETS_OU      = ['total_points']

CONTROLS_BASE   = ['close_game_epa_delta', 'close_game_def_epa_delta', 'sp_rating_delta']

CONFERENCES = sorted(paired['conference'].unique())
TIERS       = ['P4', 'G5']
TRAJECTORY_BUCKETS = ['game_1', 'games_2_4', 'games_5_8', 'games_9_12']

print("Helpers defined.")
print(f"Script features:  {SCRIPT_FEATURES}")
print(f"Controls:         {CONTROLS_BASE}")
print(f"Conferences:      {CONFERENCES}")
print(f"Trajectory buckets: {TRAJECTORY_BUCKETS}")

Helpers defined.
Script features:  ['game_script_ordinal_delta', 'game_script_avg_margin_delta', 'close_game_play_count_delta']
Controls:         ['close_game_epa_delta', 'close_game_def_epa_delta', 'sp_rating_delta']
Conferences:      ['ACC', 'American Athletic', 'Big 12', 'Big Ten', 'Conference USA', 'Mid-American', 'Mountain West', 'Pac-12', 'SEC', 'Sun Belt']
Trajectory buckets: ['game_1', 'games_2_4', 'games_5_8', 'games_9_12']


In [42]:
# Cell 6 — Full-population spread and O/U partial r table

results_full = partial_r_table(
    df       = paired,
    features = SCRIPT_FEATURES,
    targets  = TARGETS_SPREAD + TARGETS_OU,
    controls = CONTROLS_BASE,
)

print("=== Full-Population Signal Table ===")
print(f"N games: {len(paired):,}")
print()
pivot = results_full.pivot_table(
    index='feature',
    columns='target',
    values=['partial_r', 'p_value', 'n', 'signal'],
    aggfunc='first'
)
# Flat display
for feat in SCRIPT_FEATURES:
    print(f"Feature: {feat}")
    for tgt in TARGETS_SPREAD + TARGETS_OU:
        row = results_full[(results_full['feature'] == feat) & (results_full['target'] == tgt)].iloc[0]
        sig_flag = '✓ SIGNAL' if row['signal'] else '✗ no signal'
        print(f"  {tgt:<22} partial_r={row['partial_r']:>7.4f}  p={row['p_value']:.4f}  n={int(row['n']):,}  {sig_flag}")
    print()

=== Full-Population Signal Table ===
N games: 1,604

Feature: game_script_ordinal_delta
  point_differential     partial_r= 0.5673  p=0.0000  n=1,604  ✓ SIGNAL
  total_points           partial_r=-0.0371  p=0.1371  n=1,604  ✗ no signal

Feature: game_script_avg_margin_delta
  point_differential     partial_r= 0.6568  p=0.0000  n=1,604  ✓ SIGNAL
  total_points           partial_r=-0.0401  p=0.1084  n=1,604  ✗ no signal

Feature: close_game_play_count_delta
  point_differential     partial_r= 0.1834  p=0.0000  n=1,604  ✓ SIGNAL
  total_points           partial_r=-0.0298  p=0.2334  n=1,604  ✗ no signal



In [43]:
# Cell 7 — Full-population moneyline variance signal

mv_full = pd.concat([
    moneyline_variance_partial_r(paired, feat, CONTROLS_BASE)
    for feat in SCRIPT_FEATURES
], ignore_index=True)

print("=== Full-Population Moneyline Variance Table ===")
print(f"N games: {len(paired):,}")
print()
for feat in SCRIPT_FEATURES:
    row = mv_full[mv_full['feature'] == feat].iloc[0]
    sig_flag = '✓ SIGNAL' if row['signal'] else '✗ no signal'
    print(f"Feature: {feat}")
    print(f"  abs_residual_pd        partial_r={row['partial_r']:>7.4f}  p={row['p_value']:.4f}  n={int(row['n']):,}  {sig_flag}")
    print()

=== Full-Population Moneyline Variance Table ===
N games: 1,604

Feature: game_script_ordinal_delta
  abs_residual_pd        partial_r= 0.0121  p=0.6283  n=1,604  ✗ no signal

Feature: game_script_avg_margin_delta
  abs_residual_pd        partial_r=-0.0013  p=0.9572  n=1,604  ✗ no signal

Feature: close_game_play_count_delta
  abs_residual_pd        partial_r=-0.0093  p=0.7104  n=1,604  ✗ no signal



In [44]:
# Cell 8 — Retrospective diagnostic: decompose game_script signal

# game_script_avg_margin and game_script_ordinal are POST-GAME features.
# They describe the margin profile of the game that just happened.
# This cell establishes that explicitly and tests whether close_game_play_count
# — the only candidate that could have a pregame proxy — behaves differently.

# Test 1: Correlation of game_script_avg_margin_delta with point_differential (raw)
# If r is very high, it is partially tautological — large margin games produce
# large game_script_avg_margin by construction.

r_raw_script, p_raw_script = stats.pearsonr(
    paired['game_script_avg_margin_delta'].dropna(),
    paired['point_differential'].dropna()
)
print("=== Retrospective Diagnostic ===")
print()
print(f"game_script_avg_margin_delta vs point_differential (raw, no controls):")
print(f"  r = {r_raw_script:.4f}  p = {p_raw_script:.4f}")
print()

r_raw_ord, p_raw_ord = stats.pearsonr(
    paired['game_script_ordinal_delta'].dropna(),
    paired['point_differential'].dropna()
)
print(f"game_script_ordinal_delta vs point_differential (raw, no controls):")
print(f"  r = {r_raw_ord:.4f}  p = {p_raw_ord:.4f}")
print()

# Test 2: Correlation of game_script_avg_margin_delta with close_game_epa_delta
# If very high, game script is largely a restatement of EPA quality difference
r_gs_epa, p_gs_epa = stats.pearsonr(
    paired['game_script_avg_margin_delta'].dropna(),
    paired['close_game_epa_delta'].dropna()
)
print(f"game_script_avg_margin_delta vs close_game_epa_delta:")
print(f"  r = {r_gs_epa:.4f}  p = {p_gs_epa:.4f}")
print()

# Test 3: Correlation of game_script_avg_margin_delta with sp_rating_delta
r_gs_sp, p_gs_sp = stats.pearsonr(
    paired['game_script_avg_margin_delta'].dropna(),
    paired['sp_rating_delta'].dropna()
)
print(f"game_script_avg_margin_delta vs sp_rating_delta:")
print(f"  r = {r_gs_sp:.4f}  p = {p_gs_sp:.4f}")
print()

# Test 4: Does close_game_play_count_delta correlate with point_differential raw?
r_cg_pd, p_cg_pd = stats.pearsonr(
    paired['close_game_play_count_delta'].dropna(),
    paired['point_differential'].dropna()
)
print(f"close_game_play_count_delta vs point_differential (raw, no controls):")
print(f"  r = {r_cg_pd:.4f}  p = {p_cg_pd:.4f}")
print()

# Test 5: Regress point_differential on game_script_avg_margin_delta alone
# to see how much variance it explains before any controls
from sklearn.metrics import r2_score as sk_r2
sub = paired[['game_script_avg_margin_delta', 'point_differential']].dropna()
reg = LinearRegression().fit(sub[['game_script_avg_margin_delta']], sub['point_differential'])
r2_alone = sk_r2(sub['point_differential'], reg.predict(sub[['game_script_avg_margin_delta']]))
print(f"game_script_avg_margin_delta R² for point_differential (alone): {r2_alone:.4f}")
print()

# Summary
print("--- Interpretation ---")
print("game_script_avg_margin and game_script_ordinal are POST-GAME features.")
print("High raw r with point_differential confirms partial tautology.")
print("These features are DIAGNOSTIC ONLY for retrospective analysis.")
print("close_game_play_count_delta is the only candidate with a potential pregame proxy.")
print("It will be tested further in stratification cells.")

=== Retrospective Diagnostic ===

game_script_avg_margin_delta vs point_differential (raw, no controls):
  r = 0.9075  p = 0.0000

game_script_ordinal_delta vs point_differential (raw, no controls):
  r = 0.8628  p = 0.0000

game_script_avg_margin_delta vs close_game_epa_delta:
  r = 0.8172  p = 0.0000

game_script_avg_margin_delta vs sp_rating_delta:
  r = 0.6393  p = 0.0000

close_game_play_count_delta vs point_differential (raw, no controls):
  r = 0.2256  p = 0.0000

game_script_avg_margin_delta R² for point_differential (alone): 0.8235

--- Interpretation ---
game_script_avg_margin and game_script_ordinal are POST-GAME features.
High raw r with point_differential confirms partial tautology.
These features are DIAGNOSTIC ONLY for retrospective analysis.
close_game_play_count_delta is the only candidate with a potential pregame proxy.
It will be tested further in stratification cells.


In [45]:
# Cell 9 — Conference stratification: close_game_play_count_delta spread and O/U signal

LIVE_FEATURES = ['close_game_play_count_delta']

# --- Tier stratification (P4 / G5)
results_tier = partial_r_table(
    df         = paired,
    features   = LIVE_FEATURES,
    targets    = TARGETS_SPREAD + TARGETS_OU,
    controls   = CONTROLS_BASE,
    group_col  = 'tier',
    group_vals = TIERS,
)

print("=== Tier Stratification (P4 / G5) ===")
for tier in TIERS:
    print(f"\n  Tier: {tier}")
    sub = results_tier[results_tier['group'] == tier]
    for _, row in sub.iterrows():
        sig_flag = '✓ SIGNAL' if row['signal'] else '✗ no signal'
        print(f"    {row['target']:<22} partial_r={row['partial_r']:>7.4f}  p={row['p_value']:.4f}  n={int(row['n']):,}  {sig_flag}")

# --- Individual conference stratification
results_conf = partial_r_table(
    df         = paired,
    features   = LIVE_FEATURES,
    targets    = TARGETS_SPREAD + TARGETS_OU,
    controls   = CONTROLS_BASE,
    group_col  = 'conference',
    group_vals = CONFERENCES,
)

print("\n=== Individual Conference Stratification ===")
for conf in CONFERENCES:
    print(f"\n  Conference: {conf}")
    sub = results_conf[results_conf['group'] == conf]
    for _, row in sub.iterrows():
        if np.isnan(row['partial_r']):
            print(f"    {row['target']:<22} insufficient sample (n={int(row['n'])})")
        else:
            sig_flag = '✓ SIGNAL' if row['signal'] else '✗ no signal'
            print(f"    {row['target']:<22} partial_r={row['partial_r']:>7.4f}  p={row['p_value']:.4f}  n={int(row['n']):,}  {sig_flag}")

=== Tier Stratification (P4 / G5) ===

  Tier: P4
    point_differential     partial_r= 0.1760  p=0.0000  n=754  ✓ SIGNAL
    total_points           partial_r=-0.1116  p=0.0021  n=754  ✓ SIGNAL

  Tier: G5
    point_differential     partial_r= 0.1886  p=0.0000  n=850  ✓ SIGNAL
    total_points           partial_r= 0.0296  p=0.3883  n=850  ✗ no signal

=== Individual Conference Stratification ===

  Conference: ACC
    point_differential     partial_r= 0.1759  p=0.0175  n=182  ✓ SIGNAL
    total_points           partial_r=-0.0524  p=0.4822  n=182  ✗ no signal

  Conference: American Athletic
    point_differential     partial_r= 0.2522  p=0.0013  n=160  ✓ SIGNAL
    total_points           partial_r=-0.0175  p=0.8258  n=160  ✗ no signal

  Conference: Big 12
    point_differential     partial_r= 0.3099  p=0.0000  n=183  ✓ SIGNAL
    total_points           partial_r=-0.2282  p=0.0019  n=183  ✓ SIGNAL

  Conference: Big Ten
    point_differential     partial_r= 0.0878  p=0.2051  n=210  ✗ n

In [46]:
# Cell 10 — Within-season trajectory: close_game_play_count_delta

print("=== Within-Season Trajectory: close_game_play_count_delta ===")
print()

results_traj = partial_r_table(
    df         = paired,
    features   = LIVE_FEATURES,
    targets    = TARGETS_SPREAD + TARGETS_OU,
    controls   = CONTROLS_BASE,
    group_col  = 'trajectory',
    group_vals = TRAJECTORY_BUCKETS,
)

for bucket in TRAJECTORY_BUCKETS:
    print(f"  Bucket: {bucket}")
    sub = results_traj[results_traj['group'] == bucket]
    for _, row in sub.iterrows():
        if np.isnan(row['partial_r']):
            print(f"    {row['target']:<22} insufficient sample (n={int(row['n'])})")
        else:
            sig_flag = '✓ SIGNAL' if row['signal'] else '✗ no signal'
            print(f"    {row['target']:<22} partial_r={row['partial_r']:>7.4f}  p={row['p_value']:.4f}  n={int(row['n']):,}  {sig_flag}")
    print()

# --- Moneyline variance by trajectory
print("=== Moneyline Variance by Trajectory: close_game_play_count_delta ===")
print()
mv_traj = moneyline_variance_partial_r(
    paired, 'close_game_play_count_delta', CONTROLS_BASE,
    group_col='trajectory', group_vals=TRAJECTORY_BUCKETS
)
for _, row in mv_traj.iterrows():
    if np.isnan(row['partial_r']):
        print(f"  {row['group']:<14} insufficient sample (n={int(row['n'])})")
    else:
        sig_flag = '✓ SIGNAL' if row['signal'] else '✗ no signal'
        print(f"  {row['group']:<14} partial_r={row['partial_r']:>7.4f}  p={row['p_value']:.4f}  n={int(row['n']):,}  {sig_flag}")

=== Within-Season Trajectory: close_game_play_count_delta ===

  Bucket: game_1
    point_differential     partial_r= 0.1676  p=0.0219  n=187  ✓ SIGNAL
    total_points           partial_r= 0.1713  p=0.0191  n=187  ✓ SIGNAL

  Bucket: games_2_4
    point_differential     partial_r= 0.1583  p=0.0001  n=571  ✓ SIGNAL
    total_points           partial_r= 0.0173  p=0.6807  n=571  ✗ no signal

  Bucket: games_5_8
    point_differential     partial_r= 0.2016  p=0.0000  n=765  ✓ SIGNAL
    total_points           partial_r=-0.0961  p=0.0078  n=765  ✗ no signal

  Bucket: games_9_12
    point_differential     partial_r= 0.2480  p=0.0256  n=81  ✓ SIGNAL
    total_points           partial_r=-0.1015  p=0.3670  n=81  ✗ no signal

=== Moneyline Variance by Trajectory: close_game_play_count_delta ===

  game_1         partial_r= 0.0107  p=0.8849  n=187  ✗ no signal
  games_2_4      partial_r= 0.0488  p=0.2441  n=571  ✗ no signal
  games_5_8      partial_r=-0.0375  p=0.3004  n=765  ✗ no signal
  game

In [47]:
# Cell 11 — Verdict table

verdict_rows = [
    {
        'feature': 'game_script_avg_margin_delta',
        'spread_signal':    'YES — partial_r=0.6568 but RETROSPECTIVE',
        'ou_signal':        'NO',
        'ml_variance':      'NO — partial_r=-0.0013',
        'trajectory':       'N/A — diagnostic only',
        'yoy_stability':    'N/A — game-level predictor',
        'verdict':          'diagnostic_only',
        'notes': (
            'Raw r=0.9075 with point_differential. R²=0.8235 alone. '
            'Near-tautological — avg margin across game IS the score. '
            'Post-game feature only. Do not use as model input.'
        ),
    },
    {
        'feature': 'game_script_ordinal_delta',
        'spread_signal':    'YES — partial_r=0.5673 but RETROSPECTIVE',
        'ou_signal':        'NO',
        'ml_variance':      'NO — partial_r=0.0121',
        'trajectory':       'N/A — diagnostic only',
        'yoy_stability':    'N/A — game-level predictor',
        'verdict':          'diagnostic_only',
        'notes': (
            'Raw r=0.8628 with point_differential. Categories derived from '
            'in-game score margin. Post-game feature only. '
            'Do not use as model input.'
        ),
    },
    {
        'feature': 'close_game_play_count_delta',
        'spread_signal':    'YES — full_pop partial_r=0.1834, holds game_1 through games_9_12',
        'ou_signal':        'CONFERENCE-SPECIFIC — Big 12 only (r=-0.2282). game_1 bucket (r=0.1713).',
        'ml_variance':      'NO — flat across all buckets',
        'trajectory':       'Holds from game_1 (r=0.1676). Strengthens late (games_9_12 r=0.2480).',
        'yoy_stability':    'N/A — game-level predictor, not gated by YoY stability',
        'verdict':          'conference_specific_candidate',
        'notes': (
            'Spread signal holds in 6/10 conferences: ACC, American Athletic, '
            'Big 12, Mid-American, Pac-12, Sun Belt. '
            'Absent in Big Ten, Conference USA, Mountain West, SEC. '
            'Not tautological (raw r=0.2256). '
            'O/U signal Big 12 only — too narrow to generalize. '
            'Moneyline variance: no signal. '
            'Prior seed: not applicable.'
        ),
    },
]

verdict_df = pd.DataFrame(verdict_rows)

print("=== Day 17 Verdict Table ===")
print()
for _, row in verdict_df.iterrows():
    print(f"Feature:        {row['feature']}")
    print(f"Verdict:        {row['verdict']}")
    print(f"Spread:         {row['spread_signal']}")
    print(f"O/U:            {row['ou_signal']}")
    print(f"ML Variance:    {row['ml_variance']}")
    print(f"Trajectory:     {row['trajectory']}")
    print(f"YoY Stability:  {row['yoy_stability']}")
    print(f"Notes:          {row['notes']}")
    print()

# --- Save verdict CSV
verdict_df.to_csv('artifacts/game_script_verdict.csv', index=False)
print("Saved: artifacts/game_script_verdict.csv")

=== Day 17 Verdict Table ===

Feature:        game_script_avg_margin_delta
Verdict:        diagnostic_only
Spread:         YES — partial_r=0.6568 but RETROSPECTIVE
O/U:            NO
ML Variance:    NO — partial_r=-0.0013
Trajectory:     N/A — diagnostic only
YoY Stability:  N/A — game-level predictor
Notes:          Raw r=0.9075 with point_differential. R²=0.8235 alone. Near-tautological — avg margin across game IS the score. Post-game feature only. Do not use as model input.

Feature:        game_script_ordinal_delta
Verdict:        diagnostic_only
Spread:         YES — partial_r=0.5673 but RETROSPECTIVE
O/U:            NO
ML Variance:    NO — partial_r=0.0121
Trajectory:     N/A — diagnostic only
YoY Stability:  N/A — game-level predictor
Notes:          Raw r=0.8628 with point_differential. Categories derived from in-game score margin. Post-game feature only. Do not use as model input.

Feature:        close_game_play_count_delta
Verdict:        conference_specific_candidate
Spread

In [48]:
# Cell 12 — Final markdown summary

summary = """
# Day 17 — Game Script Analysis: Final Summary

## Population
- 1,607 valid FBS conference games (2022–2024)
- 3,214 team-game rows; 1,604 analysis games after dropping 6 null-control rows
- FBS Independents excluded; both-team FBS integrity confirmed
- Controls: close_game_epa_delta, close_game_def_epa_delta, sp_rating_delta

---

## Candidates Tested
1. `game_script_avg_margin_delta`
2. `game_script_ordinal_delta`
3. `close_game_play_count_delta`

---

## Finding 1 — game_script_avg_margin_delta: diagnostic_only

**Spread:** Partial r=0.6568 after controls — but RETROSPECTIVE. Raw r=0.9075
with point_differential. R²=0.8235 alone. Average margin across the game IS
the score by construction. This is near-tautological, not predictive signal.

**O/U:** No signal. Partial r=-0.0401 (p=0.1084).

**Moneyline variance:** No signal. Partial r=-0.0013 (p=0.9572).

**Verdict: diagnostic_only.** Post-game feature. Do not use as model input.
Useful only for retrospective decomposition of outcomes.

---

## Finding 2 — game_script_ordinal_delta: diagnostic_only

**Spread:** Partial r=0.5673 after controls — but RETROSPECTIVE. Raw r=0.8628
with point_differential. Ordinal categories (dominant/comfortable/competitive/
deficit/large_deficit) are derived from in-game score margin.

**O/U:** No signal. Partial r=-0.0371 (p=0.1371).

**Moneyline variance:** No signal. Partial r=0.0121 (p=0.6283).

**Verdict: diagnostic_only.** Post-game feature. Do not use as model input.

---

## Finding 3 — close_game_play_count_delta: conference_specific_candidate

**Spread:** Full-population partial r=0.1834 (p<0.0001). Not tautological
(raw r=0.2256). Signal holds in 6/10 conferences:
- ACC (r=0.1759), American Athletic (r=0.2522), Big 12 (r=0.3099),
  Mid-American (r=0.1733), Pac-12 (r=0.3079), Sun Belt (r=0.2108)

No signal in: Big Ten, Conference USA, Mountain West, SEC.
No clean tier split — non-signal conferences span both P4 and G5.

**O/U:** Conference-specific only. Big 12 r=-0.2282 (p=0.0019) — negative,
meaning more close-game plays associates with lower totals in that conference.
game_1 bucket r=0.1713 (p=0.0191) — narrow and trajectory-specific.
Too isolated to generalize.

**Moneyline variance:** No signal across full population or any trajectory bucket.

**Within-season trajectory:**
- game_1:    spread r=0.1676 (p=0.0219) — signal present at conf game 1
- games_2_4: spread r=0.1583 (p=0.0001)
- games_5_8: spread r=0.2016 (p<0.0001)
- games_9_12: spread r=0.2480 (p=0.0256)
Signal holds from game_1 and strengthens monotonically. Available at all
information states — does not require rolling data to activate.

**YoY stability:** Not applicable — game-level predictor, not gated by YoY.

**Verdict: conference_specific_candidate** for spread.
Not valid for O/U (except Big 12), moneyline variance, or prior seeding.

---

## Key Methodological Note

game_script_avg_margin and game_script_ordinal are POST-GAME features.
Their high partial r after controls is residual tautology, not predictive power.
They describe the game that happened, not information available pregame.
close_game_play_count is the only game-script candidate that is not inherently
retrospective — it accumulates across plays during the game and could support
a rolling prior-season version for pregame deployment.

---

## Verdict Summary

| Feature                        | Spread | O/U | ML Variance | Verdict                      |
|-------------------------------|--------|-----|-------------|------------------------------|
| game_script_avg_margin_delta  | RETRO  | NO  | NO          | diagnostic_only              |
| game_script_ordinal_delta     | RETRO  | NO  | NO          | diagnostic_only              |
| close_game_play_count_delta   | YES*   | NO† | NO          | conference_specific_candidate|

*Spread: 6/10 conferences. Holds from game_1.
†O/U: Big 12 only (r=-0.2282). game_1 bucket only elsewhere.

---

## Artifact
`artifacts/game_script_verdict.csv` — saved.
"""

print(summary)


# Day 17 — Game Script Analysis: Final Summary

## Population
- 1,607 valid FBS conference games (2022–2024)
- 3,214 team-game rows; 1,604 analysis games after dropping 6 null-control rows
- FBS Independents excluded; both-team FBS integrity confirmed
- Controls: close_game_epa_delta, close_game_def_epa_delta, sp_rating_delta

---

## Candidates Tested
1. `game_script_avg_margin_delta`
2. `game_script_ordinal_delta`
3. `close_game_play_count_delta`

---

## Finding 1 — game_script_avg_margin_delta: diagnostic_only

**Spread:** Partial r=0.6568 after controls — but RETROSPECTIVE. Raw r=0.9075
with point_differential. R²=0.8235 alone. Average margin across the game IS
the score by construction. This is near-tautological, not predictive signal.

**O/U:** No signal. Partial r=-0.0401 (p=0.1084).

**Moneyline variance:** No signal. Partial r=-0.0013 (p=0.9572).

**Verdict: diagnostic_only.** Post-game feature. Do not use as model input.
Useful only for retrospective decomposition of outcom